In [0]:
# ==========================================
# 1. WARSTWA GOLD - Agregacje biznesowe
# ==========================================
print("⏳ Budowanie tabeli Gold (Gotowej do podpięcia pod Power BI) w ADLS Gen2...")

# Używamy bezpiecznego DROP TABLE z opcją usunięcia danych lub bezpiecznego nadpisania
spark.sql("DROP TABLE IF EXISTS dbw_showcase.default.payroll_gold")

# Jeśli fizyczny folder w chmurze trzyma stare pliki, najbezpieczniej utworzyć tabelę jako Managed w External Location 
# lub użyć REPLACE TABLE ze wskazaniem na folder
spark.sql("""
CREATE OR REPLACE TABLE dbw_showcase.default.payroll_gold 
LOCATION 'abfss://gold@adlsportfolioaw2026.dfs.core.windows.net/tables/payroll_gold'
AS
SELECT 
    h.department_name,
    h.city,
    COUNT(p.transaction_id) AS total_transactions,
    ROUND(SUM(p.hours_logged), 2) AS total_hours
FROM dbw_showcase.default.payroll_silver p
JOIN dbw_showcase.default.hr_silver h 
  ON p.emp_id = h.emp_id
WHERE h.is_current = true
GROUP BY h.department_name, h.city
""")
print("✅ Tabela Gold utworzona pomyślnie w kontenerze Gold!")

# ==========================================
# 2. DELTA MAINTENANCE: OPTIMIZE & Z-ORDER
# ==========================================
print("⏳ Optymalizacja fizycznego składowania danych (Small File Problem)...")
spark.sql("OPTIMIZE dbw_showcase.default.payroll_silver ZORDER BY (emp_id)")
print("✅ Tabela Silver zoptymalizowana (OPTIMIZE + Z-ORDER)!")

# ==========================================
# 3. GDPR COMPLIANCE: Prawo do bycia zapomnianym (DELETE + VACUUM)
# ==========================================
print("⏳ Wykonywanie usunięcia zgodnego z GDPR...")
# Krok A: Usuwamy pracownika z logiki biznesowej
spark.sql("DELETE FROM dbw_showcase.default.hr_silver WHERE emp_id = 10102")

# Krok B: VACUUM dla Delta Lake (w środowisku Serverless korzystamy z domyślnej polityki retencji)
spark.sql("VACUUM dbw_showcase.default.hr_silver")

print("✅ Proces GDPR zrealizowany pomyślnie!")